In [ ]:
from neuron import h

def resample_segments(cell_root, dL_target: float):
    """
    Traverse cell_root.subtree(), read each section's length (µm), 
    and reset its nseg so that sec.L / nseg ≈ dL_target. 
    Ensures nseg is odd. Returns a dict of {sec_name: old_nseg -> new_nseg}.
    """
    # 1) Make sure sec.L is correct for every section:
    h.define_shape()

    changed = {}
    for sec in cell_root.subtree():
        L = sec.L                      # total physical length of this section
        if L <= 0:
            continue                   # guard against degenerate sections
        
        # 2) compute nearest integer for sec.L / dL_target
        nseg_new = int(L / dL_target + 0.5)
        if nseg_new < 1:
            nseg_new = 1              # minimum of 1 segment
        # 3) force odd
        if nseg_new % 2 == 0:
            nseg_new += 1

        old_nseg = sec.nseg
        if old_nseg != nseg_new:
            changed[sec.name()] = (old_nseg, nseg_new)
            sec.nseg = nseg_new

    # 4) If you want to update any derived geometry right away:
    h.define_shape()
    return changed

# Example usage:
#  Say you want every segment to be ~ 15 µm long,
#  and "root" is cell.dend[0] (or whatever branch you’re resampling).
root = cell.dend[0]
dL_target = 15.0  # µm

diffs = resample_segments(root, dL_target)
print("Sections whose nseg changed (old → new):")
for secname, (old, new) in diffs.items():
    print(f"  {secname:20s} : {old} → {new}")


In [ ]:
h.load_file("stdlib.hoc")
h.celsius = 36  # e.g. for temperature‐dependent lambda
d_lambda = 0.1  # compartments per 0.1 λ

for sec in cell.dend[0].subtree():
    lambda_f = h.lambda_f  # NEURON’s built‐in frequency (100 Hz by default)
    nseg = int((sec.L / (d_lambda * lambda_f)) + 0.9)
    if nseg % 2 == 0:
        nseg += 1
    sec.nseg = nseg
h.define_shape()
